# 02 - Data Preprocessing: Chip Companies Financials

This notebook preprocesses a copy of the raw CSV; the raw file is never modified. Confirmed errors and diagnostic flags are treated differently, and no outliers are removed without supporting evidence.

In [1]:
from pathlib import Path
import pandas as pd

## Load the raw data

In [2]:
cwd = Path.cwd().resolve()
project_root = cwd if (cwd / "data" / "raw").is_dir() else cwd.parent
input_path = project_root / "data" / "raw" / "chip_companies_financials.csv"
output_path = Path('C:\\Users\\ALBERT\\Desktop\\Year 1 sem2\\DADS5001_Aj.Thitirat_SUN\\Mid-Term Project\\semiconductor company datasets\\outputs\\clean_eda_2026_09_11\\financials_clean.csv')
if not input_path.is_file():
    raise FileNotFoundError(f"Raw dataset not found: {input_path}")
if not output_path.parent.is_dir():
    raise FileNotFoundError(f"Processed-data directory not found: {output_path.parent}")

df_raw = pd.read_csv(input_path)
df_clean = df_raw.copy()
df_raw_snapshot = df_raw.copy(deep=True)
print(f"Input path: {input_path}")
print(f"Original shape: {df_raw.shape}")
print(f"Original columns: {df_raw.columns.tolist()}")
print(f"Original year range: {df_raw['year'].min()}–{df_raw['year'].max()}")

Input path: C:\Users\ALBERT\Desktop\Year 1 sem2\DADS5001_Aj.Thitirat_SUN\Mid-Term Project\semiconductor company datasets\data\raw\chip_companies_financials.csv
Original shape: (617, 10)
Original columns: ['year', 'company_name', 'ticker', 'country_iso3', 'segment', 'revenue_usd_bn', 'operating_margin_pct', 'operating_income_usd_bn', 'rd_spend_usd_bn', 'capex_usd_bn']
Original year range: 2010–2026


## Standardize types and trim text

Text values are changed only by removing leading and trailing whitespace.

In [3]:
text_columns = ["company_name", "ticker", "country_iso3", "segment"]
financial_columns = ["revenue_usd_bn", "operating_margin_pct", "operating_income_usd_bn", "rd_spend_usd_bn", "capex_usd_bn"]

df_clean["year"] = pd.to_numeric(df_clean["year"], errors="raise").astype("int64")
for column in financial_columns:
    df_clean[column] = pd.to_numeric(df_clean[column], errors="raise")

text_change_counts = {}
for column in text_columns:
    before = df_clean[column].astype("string")
    after = before.str.strip()
    text_change_counts[column] = int(before.ne(after).fillna(False).sum())
    df_clean[column] = after
display(pd.Series(text_change_counts, name="values_changed").to_frame())
display(df_clean.dtypes.rename("dtype").to_frame())

,values_changed
company_name,0
ticker,0
country_iso3,0
segment,0


,dtype
year,int64
company_name,string[python]
ticker,string[python]
country_iso3,string[python]
segment,string[python]
revenue_usd_bn,float64
operating_margin_pct,float64
operating_income_usd_bn,float64
rd_spend_usd_bn,float64
capex_usd_bn,float64


## Geography and time flags

EUR is preserved exactly as supplied and flagged as a regional code. The 2025–2026 caution flag does not prove that these observations are forecasts; it marks them for source validation.

Project-defined periods: 2010–2021 = Smart automobile expansion; 2022 = Transition Year; 2023–2026 = AI expansion. These labels organize comparisons and do not establish the cause of financial changes.

In [4]:
eur_mask = df_clean["country_iso3"].eq("EUR")
df_clean["geo_code_type"] = "country"
df_clean.loc[eur_mask, "geo_code_type"] = "region"
df_clean["country_code_review_flag"] = eur_mask

df_clean["analysis_period"] = pd.Series(pd.NA, index=df_clean.index, dtype="string")
df_clean.loc[df_clean["year"].between(2010, 2021), "analysis_period"] = "Smart automobile expansion"
df_clean.loc[df_clean["year"].eq(2022), "analysis_period"] = "Transition Year"
df_clean.loc[df_clean["year"].between(2023, 2026), "analysis_period"] = "AI expansion"
assert df_clean["analysis_period"].notna().all(), "Validation failed: at least one row has no analysis_period."
assert df_clean["analysis_period"].isin(["Smart automobile expansion", "Transition Year", "AI expansion"]).all(), "Validation failed: unexpected analysis_period value."
df_clean["source_validation_caution"] = df_clean["year"].isin([2025, 2026])
display(df_clean[["country_iso3", "geo_code_type", "country_code_review_flag"]].value_counts().rename("row_count").to_frame())
display(df_clean["analysis_period"].value_counts(dropna=False).rename("row_count").to_frame())
display(df_clean["source_validation_caution"].value_counts(dropna=False).rename("row_count").to_frame())

,,,row_count
country_iso3,geo_code_type,country_code_review_flag,
USA,country,False,318
CHN,country,False,70
JPN,country,False,51
KOR,country,False,51
TWN,country,False,51
NLD,country,False,34
DEU,country,False,17
EUR,region,True,17
GBR,country,False,8


,row_count
analysis_period,
Smart automobile expansion,417
AI expansion,160
Transition Year,40


,row_count
source_validation_caution,
False,537
True,80


## Operating-income diagnostics and financial intensity

Reported operating income is preserved. Ratios are calculated only for positive revenue, so zero or negative revenue cannot produce infinity. Revenue growth is intentionally deferred to feature engineering.

In [5]:
df_clean["calculated_operating_income_usd_bn"] = df_clean["revenue_usd_bn"] * df_clean["operating_margin_pct"] / 100
df_clean["operating_income_abs_diff_usd_bn"] = (df_clean["operating_income_usd_bn"] - df_clean["calculated_operating_income_usd_bn"]).abs()
df_clean["operating_income_review_flag"] = df_clean["operating_income_abs_diff_usd_bn"].gt(0.02)
df_clean["zero_revenue_flag"] = df_clean["revenue_usd_bn"].eq(0)
positive_revenue = df_clean["revenue_usd_bn"].where(df_clean["revenue_usd_bn"].gt(0))
df_clean["rd_intensity_pct"] = df_clean["rd_spend_usd_bn"].div(positive_revenue).mul(100)
df_clean["capex_intensity_pct"] = df_clean["capex_usd_bn"].div(positive_revenue).mul(100)
display(df_clean[["calculated_operating_income_usd_bn", "operating_income_abs_diff_usd_bn", "rd_intensity_pct", "capex_intensity_pct"]].describe().T)
print(f"Operating-income review rows: {df_clean['operating_income_review_flag'].sum()}")
print(f"Zero-revenue rows: {df_clean['zero_revenue_flag'].sum()}")

,count,mean,std,min,25%,50%,75%,max
calculated_operating_income_usd_bn,617.0,4.504825,9.881256,0.000000,1.076790,2.083560,4.910520,155.170920
operating_income_abs_diff_usd_bn,617.0,0.004659,0.006525,0.000000,0.001400,0.003080,0.005210,0.079080
rd_intensity_pct,609.0,15.143145,10.379782,7.352941,10.008028,11.995104,12.086093,100.000000
capex_intensity_pct,609.0,18.394012,15.068915,0.000000,9.976976,10.071942,34.983790,45.714286


Operating-income review rows: 18
Zero-revenue rows: 8


## Arrange columns and validate before saving

In [6]:
final_columns = [
    "year", "company_name", "ticker", "country_iso3", "geo_code_type",
    "country_code_review_flag", "segment", "analysis_period", "source_validation_caution",
    "revenue_usd_bn", "operating_margin_pct", "operating_income_usd_bn", "rd_spend_usd_bn", "capex_usd_bn",
    "calculated_operating_income_usd_bn", "operating_income_abs_diff_usd_bn", "operating_income_review_flag",
    "rd_intensity_pct", "capex_intensity_pct", "zero_revenue_flag",
]
df_clean = df_clean[final_columns]

validation_results = {}
validation_results["row_count_617"] = len(df_clean) == 617
validation_results["unique_year_company"] = not df_clean.duplicated(["year", "company_name"]).any()
validation_results["original_financial_columns_complete"] = not df_clean[financial_columns].isna().any().any()
validation_results["nonnegative_revenue_rd_capex"] = not df_clean[["revenue_usd_bn", "rd_spend_usd_bn", "capex_usd_bn"]].lt(0).any().any()
validation_results["all_periods_assigned"] = df_clean["analysis_period"].notna().all()
validation_results["eur_rows_17"] = int(df_clean["country_code_review_flag"].sum()) == 17
validation_results["operating_income_review_rows_18"] = int(df_clean["operating_income_review_flag"].sum()) == 18
validation_results["no_infinite_values"] = not df_clean.select_dtypes(include="number").isin([float("inf"), float("-inf")]).any().any()

pd.testing.assert_frame_equal(df_clean[financial_columns].reset_index(drop=True), df_raw[financial_columns].reset_index(drop=True), check_dtype=False, obj="original financial columns")
validation_results["original_financial_values_preserved"] = True
pd.testing.assert_frame_equal(df_raw, df_raw_snapshot, obj="df_raw")
validation_results["df_raw_unchanged"] = True
for check_name, passed in validation_results.items():
    assert passed, f"Validation failed: {check_name}"
display(pd.Series(validation_results, name="passed").to_frame())

,passed
row_count_617,True
unique_year_company,True
original_financial_columns_complete,True
nonnegative_revenue_rd_capex,True
all_periods_assigned,True
eur_rows_17,True
operating_income_review_rows_18,True
no_infinite_values,True
original_financial_values_preserved,True
df_raw_unchanged,True


## Preprocessing summary

In [7]:
preprocessing_summary = pd.DataFrame([
    ["Data types", len(df_clean), "Applied explicit integer, string, and numeric types", "Provide consistent analysis types; unexpected numeric values raise errors", "PASS"],
    ["Text whitespace", sum(text_change_counts.values()), "Trimmed leading and trailing whitespace only", "Conservative text standardization", "PASS"],
    ["EUR geography", int(eur_mask.sum()), "Preserved EUR; added region type and review flag", "Avoid guessing a country", "PASS"],
    ["Analysis period", len(df_clean), "Assigned project-defined period", "Support period comparisons", "PASS"],
    ["Source-validation caution", int(df_clean["source_validation_caution"].sum()), "Flagged 2025–2026 only", "These years require source validation; this does not prove forecast status", "PASS"],
    ["Operating-income diagnostic", int(df_clean["operating_income_review_flag"].sum()), "Added calculation, absolute difference, and >0.02 review flag", "Preserve reported values while exposing discrepancies", "PASS"],
    ["Financial intensity", int(df_clean["revenue_usd_bn"].gt(0).sum()), "Calculated R&D and CapEx percentages for positive revenue", "Avoid infinite ratios", "PASS"],
    ["Zero revenue", int(df_clean["zero_revenue_flag"].sum()), "Added flag; ratios remain missing", "Avoid division by zero", "PASS"],
    ["Rows and extremes", 0, "Removed no rows and no statistical extremes", "No supporting evidence justifies removal", "PASS"],
], columns=["transformation", "affected_rows", "action_taken", "reason", "validation_result"])
display(preprocessing_summary)

,transformation,affected_rows,action_taken,reason,validation_result
0,Data types,617,"Applied explicit integer, string, and numeric ...",Provide consistent analysis types; unexpected ...,PASS
1,Text whitespace,0,Trimmed leading and trailing whitespace only,Conservative text standardization,PASS
2,EUR geography,17,Preserved EUR; added region type and review flag,Avoid guessing a country,PASS
3,Analysis period,617,Assigned project-defined period,Support period comparisons,PASS
4,Source-validation caution,80,Flagged 2025–2026 only,These years require source validation; this do...,PASS
5,Operating-income diagnostic,18,"Added calculation, absolute difference, and >0...",Preserve reported values while exposing discre...,PASS
6,Financial intensity,609,Calculated R&D and CapEx percentages for posit...,Avoid infinite ratios,PASS
7,Zero revenue,8,Added flag; ratios remain missing,Avoid division by zero,PASS
8,Rows and extremes,0,Removed no rows and no statistical extremes,No supporting evidence justifies removal,PASS


## Save and re-read validation

In [8]:
df_clean.to_csv(output_path, index=False, encoding="utf-8")
df_saved = pd.read_csv(output_path)
saved_validation = {
    "shape": df_saved.shape == df_clean.shape,
    "column_order": df_saved.columns.tolist() == final_columns,
    "year_range": (int(df_saved["year"].min()), int(df_saved["year"].max())) == (2010, 2026),
    "duplicate_year_company_keys": int(df_saved.duplicated(["year", "company_name"]).sum()) == 0,
    "original_financial_missing_values": int(df_saved[financial_columns].isna().sum().sum()) == 0,
    "eur_flags": int(df_saved["country_code_review_flag"].sum()) == 17,
    "operating_income_review_flags": int(df_saved["operating_income_review_flag"].sum()) == 18,
}
for check_name, passed in saved_validation.items():
    assert passed, f"Saved-file validation failed: {check_name}"
print(f"Saved output: {output_path}")
print(f"Re-read shape: {df_saved.shape}")
print(f"Re-read year range: {df_saved['year'].min()}–{df_saved['year'].max()}")
print(f"Duplicate year-company keys: {df_saved.duplicated(['year', 'company_name']).sum()}")
display(df_saved.isna().sum().rename("missing_count").to_frame())
print(f"EUR flags: {df_saved['country_code_review_flag'].sum()}")
print(f"Operating-income review flags: {df_saved['operating_income_review_flag'].sum()}")
display(pd.Series(saved_validation, name="passed").to_frame())

Saved output: C:\Users\ALBERT\Desktop\Year 1 sem2\DADS5001_Aj.Thitirat_SUN\Mid-Term Project\semiconductor company datasets\outputs\clean_eda_2026_09_11\financials_clean.csv
Re-read shape: (617, 20)
Re-read year range: 2010–2026
Duplicate year-company keys: 0


,missing_count
year,0
company_name,0
ticker,0
country_iso3,0
geo_code_type,0
country_code_review_flag,0
segment,0
analysis_period,0
source_validation_caution,0
revenue_usd_bn,0


EUR flags: 17
Operating-income review flags: 18


,passed
shape,True
column_order,True
year_range,True
duplicate_year_company_keys,True
original_financial_missing_values,True
eur_flags,True
operating_income_review_flags,True


## Preprocessing Conclusions

No rows were removed. EUR was preserved and flagged for review, and reported operating income was preserved alongside diagnostic columns. Statistical extremes were not removed. Rows from 2025–2026 were flagged only for source validation; this does not establish that they are forecasts. After all pre-save and re-read validations passed, the processed dataset is ready for feature engineering.